<a href="https://colab.research.google.com/github/Rakshaksa/IN226062302_GenAI/blob/main/Task_3_Build_a_Chatbot_using_Hugging_Face_Transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 NOVA - AI Chatbot using Hugging Face Transformers
---
**Model Used:** `microsoft/DialoGPT-medium`  
**Libraries:** Transformers, PyTorch, Accelerate

In [1]:
# Install required libraries

!pip install transformers torch accelerate

In [2]:
# Import torch for tensor operations (joining chat history)
import torch

# Import AutoModelForCausalLM: loads the language generation model
# Import AutoTokenizer: converts text to tokens and back
from transformers import AutoModelForCausalLM, AutoTokenizer

In [3]:
# Tokenizer converts user text into numbers the model can understand
tokenizer = AutoTokenizer.from_pretrained('microsoft/DialoGPT-medium')

# Load the pre-trained DialoGPT model
model = AutoModelForCausalLM.from_pretrained('microsoft/DialoGPT-medium')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/863M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/863M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [6]:
# Display welcome message when chatbot starts
print("Hey there! I'm NOVA 🤖 Your personal AI assistant. Ask me anything!")
print("-" * 60)

chat_history_ids = None

while True:

    # Take user input from the console
    text = input("You: ")

    # Check if user wants to exit the chatbot
    if text == 'exit' or text == 'close':
        print("NOVA signing off... Until next time! 👋🤖")
        break

    # Encode user input: convert text → tokens → numbers
    new_user_input_ids = tokenizer.encode(text + tokenizer.eos_token, return_tensors='pt')

    # Maintain conversation context
    if chat_history_ids is None:
        bot_input_ids = new_user_input_ids
    else:
        # torch.cat joins chat history and new input into one tensor
        bot_input_ids = torch.cat([chat_history_ids, new_user_input_ids], dim=-1)

    # Generate a response using the DialoGPT model
    chat_history_ids = model.generate(
    bot_input_ids,
    max_length=1000,
    pad_token_id=tokenizer.eos_token_id,
    no_repeat_ngram_size=3,
    do_sample=True,
    top_k=50,
    top_p=0.9,
    temperature=0.7
)

    # Decode the generated response: convert numbers → tokens → text
    output = tokenizer.decode(chat_history_ids[:, bot_input_ids.shape[-1]:][0], skip_special_tokens=True)

    # Display the chatbot response
    print("Chatbot [NOVA]:", output)
    print("-" * 60)

Hey there! I'm NOVA 🤖 Your personal AI assistant. Ask me anything!
------------------------------------------------------------
You: hello
Chatbot [NOVA]: You're good people.
------------------------------------------------------------
You: What is Artificial Intelligence?
Chatbot [NOVA]: Something you can't really understand.
------------------------------------------------------------
You: tell me i want to understand
Chatbot [NOVA]: I'm good.
------------------------------------------------------------
You: Who created Python?  
Chatbot [NOVA]: It was created by the person who created Python.
------------------------------------------------------------
You: What is your favorite food?
Chatbot [NOVA]: I don't know, but I like pizza.
------------------------------------------------------------
You: Do you think AI is dangerous?
Chatbot [NOVA]: Yes, he is a dangerous man.
------------------------------------------------------------
You: exit
NOVA signing off... Until next time! 👋🤖


NOVA is powered by DialoGPT-medium, a transformer model trained on real Reddit conversations. Because of this, it responds naturally and casually rather than giving textbook-perfect answers. Every response is 100% AI-generated — no hardcoded replies, just pure transformer magic! 🤖